In [0]:
print("=== CLIMA ===")
spark.sql("SELECT DISTINCT departamento FROM bronze.clima_raw ORDER BY departamento").show(truncate=False)

In [0]:
print("=== VENTAS ===")
spark.sql("SELECT DISTINCT departamento_destino FROM gold.resumen_ventas_mensual ORDER BY departamento_destino").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

ventas = spark.table("gold.resumen_ventas_mensual")

ventas_mes_depto = (
    ventas
    .withColumn("anio", F.year("mes"))
    .withColumn("mes_num", F.month("mes"))
    .groupBy("departamento_destino", "anio", "mes_num")
    .agg(F.round(F.sum("tm_vendidas"), 1).alias("tm_vendidas_total"))
)

ventas_mes_depto.orderBy("anio", "mes_num", "departamento_destino").show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

ventas_mes_depto = (
    spark.table("gold.resumen_ventas_mensual")
    .withColumn("anio", F.year("mes"))
    .withColumn("mes_num", F.month("mes"))
    .groupBy("departamento_destino", "anio", "mes_num")
    .agg(F.round(F.sum("tm_vendidas"), 1).alias("tm_vendidas_total"))
)

clima = (
    spark.table("bronze.clima_raw")
    .select("departamento", "anio", "mes", "precip_pronosticada_mm")
)

dataset = (
    ventas_mes_depto.join(
        clima,
        on=[
            ventas_mes_depto.departamento_destino == clima.departamento,
            ventas_mes_depto.anio == clima.anio,
            ventas_mes_depto.mes_num == clima.mes,
        ],
        how="inner"   
    )
    .select(
        ventas_mes_depto.departamento_destino,
        ventas_mes_depto.anio,
        ventas_mes_depto.mes_num,
        clima.precip_pronosticada_mm,   
        ventas_mes_depto.tm_vendidas_total,  
    )
)

print("Filas en el dataset final:", dataset.count())
dataset.orderBy("anio", "mes_num", "departamento_destino").show(12, truncate=False)

In [0]:
df = dataset.select("precip_pronosticada_mm", "tm_vendidas_total").toPandas()

print("Forma:", df.shape)

df.head()

In [0]:
from sklearn.model_selection import train_test_split

X = df[["precip_pronosticada_mm"]]

y = df["tm_vendidas_total"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      
    random_state=42     
)

print("Entrenamiento:", X_train.shape[0], "filas")
print("Prueba:", X_test.shape[0], "filas")

In [0]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()

modelo.fit(X_train, y_train)

print("Fórmula aprendida:")
print(f"  tm_vendidas = {modelo.coef_[0]:.2f} * lluvia + {modelo.intercept_:.2f}")

In [0]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = modelo.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²:  {r2:.3f}")
print(f"MAE: {mae:.1f} toneladas")

In [0]:

df = dataset.select(
    "departamento_destino",
    "precip_pronosticada_mm",
    "tm_vendidas_total"
).toPandas()

print("Forma:", df.shape)  
df.head()

In [0]:
import pandas as pd

df_encoded = pd.get_dummies(df, columns=["departamento_destino"], dtype=int)

print("Columnas ahora:", list(df_encoded.columns))
df_encoded.head()

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df_encoded.drop(columns=["tm_vendidas_total"])
y = df_encoded["tm_vendidas_total"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo2 = LinearRegression()
modelo2.fit(X_train, y_train)

y_pred = modelo2.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²:  {r2:.3f}   (antes: 0.263)")
print(f"MAE: {mae:.1f} toneladas   (antes: 1446.8)")

In [0]:

datos = dataset.select(
    "precip_pronosticada_mm",
    "tm_vendidas_total"
).toPandas()

datos_real = dataset.join(
    spark.table("bronze.clima_raw").select(
        F.col("departamento").alias("dep_c"),
        F.col("anio").alias("anio_c"),
        F.col("mes").alias("mes_c"),
        "precip_real_mm"
    ),
    on=[
        dataset.departamento_destino == F.col("dep_c"),
        dataset.anio == F.col("anio_c"),
        dataset.mes_num == F.col("mes_c"),
    ]
).select("precip_pronosticada_mm", "precip_real_mm", "tm_vendidas_total").toPandas()

print("Correlación PRONOSTICADA vs ventas:", datos_real["precip_pronosticada_mm"].corr(datos_real["tm_vendidas_total"]).round(3))
print("Correlación REAL vs ventas:", datos_real["precip_real_mm"].corr(datos_real["tm_vendidas_total"]).round(3))

In [0]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

X = df_encoded.drop(columns=["tm_vendidas_total"])
y = df_encoded["tm_vendidas_total"]

scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")

print("R² en cada una de las 5 particiones:", scores.round(3))
print(f"R² promedio: {scores.mean():.3f}")

In [0]:
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import pandas as pd

ventas_mes_depto = (
    spark.table("gold.resumen_ventas_mensual")
    .withColumn("anio", F.year("mes"))
    .withColumn("mes_num", F.month("mes"))
    .groupBy("departamento_destino", "anio", "mes_num")
    .agg(F.round(F.sum("tm_vendidas"), 1).alias("tm_vendidas_total"))
)

clima = spark.table("bronze.clima_raw").select(
    "departamento", "anio", "mes", "precip_pronosticada_mm"
)

dataset = (
    ventas_mes_depto.join(
        clima,
        on=[
            ventas_mes_depto.departamento_destino == clima.departamento,
            ventas_mes_depto.anio == clima.anio,
            ventas_mes_depto.mes_num == clima.mes,
        ],
        how="inner"
    )
    .select(
        ventas_mes_depto.departamento_destino,
        ventas_mes_depto.anio,
        ventas_mes_depto.mes_num,
        clima.precip_pronosticada_mm,
        ventas_mes_depto.tm_vendidas_total,
    )
)


df = dataset.select(
    "departamento_destino", "precip_pronosticada_mm", "tm_vendidas_total"
).toPandas()
df_encoded = pd.get_dummies(df, columns=["departamento_destino"], dtype=int)

X = df_encoded.drop(columns=["tm_vendidas_total"])
y = df_encoded["tm_vendidas_total"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Modelo reentrenado. Features:", list(X.columns))

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import r2_score, mean_absolute_error

# Iniciar un "run": todo lo que registres queda agrupado bajo este experimento
with mlflow.start_run(run_name="regresion_demanda_fertilizantes"):

    # Calcular métricas sobre el test
    y_pred = modelo.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    # Registrar parámetros (qué usamos)
    mlflow.log_param("modelo", "LinearRegression")
    mlflow.log_param("features", list(X.columns))

    # Registrar métricas (qué tan bueno fue)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)

    # Guardar el modelo entrenado como artefacto
    mlflow.sklearn.log_model(modelo, "modelo_demanda")

    print(f"Registrado en MLflow -> R²={r2:.3f}, MAE={mae:.1f}")

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.model_selection import cross_val_score

with mlflow.start_run(run_name="regresion_demanda_v2"):

    r2_cv = cross_val_score(modelo, X, y, cv=5, scoring="r2").mean()

    firma = infer_signature(X_train, modelo.predict(X_train))

    mlflow.log_param("modelo", "LinearRegression")
    mlflow.log_param("features", list(X.columns))
    mlflow.log_metric("r2_cv", r2_cv)

    mlflow.sklearn.log_model(
        modelo,
        name="modelo_demanda",
        signature=firma,             
        input_example=X_train.head(3) 
    )

    print(f"Modelo v2 guardado -> R² (validación cruzada): {r2_cv:.3f}")

In [0]:
import mlflow

modelo_prod = mlflow.sklearn.load_model("models:/m-acb0765edff343acbf3f900c60145ae5")

print("Modelo cargado desde MLflow ✓")
print("Predicción de prueba:", modelo_prod.predict(X_test.head(1)))

In [0]:
import pandas as pd


ejemplo = pd.DataFrame([{
    "precip_pronosticada_mm": 150.0,
    "departamento_destino_Antioquia": 0,
    "departamento_destino_Narino": 0,
    "departamento_destino_Quindio": 0,
    "departamento_destino_Tolima": 0,
    "departamento_destino_Valle": 1,   
}])

prediccion = modelo_prod.predict(ejemplo)
print(f"Si en Valle se pronostican 150mm de lluvia → demanda estimada: {prediccion[0]:.1f} toneladas")

In [0]:
from pyspark.sql import functions as F
import pandas as pd

# === NOTEBOOK DE PRODUCCIÓN: predecir demanda a partir del pronóstico de lluvia ===

# 1. lee el proostico de lalluvia
clima_pd = (
    spark.table("bronze.clima_raw")
    .select("departamento", "anio", "mes", "precip_pronosticada_mm")
    .toPandas()
)

# 2. se aplica la misma transfomración
clima_encoded = pd.get_dummies(clima_pd, columns=["departamento"], dtype=int)

# 3. Alinear columnas con lo que el modelo espera (mismas 6 features, mismo orden)
features_modelo = [
    "precip_pronosticada_mm",
    "departamento_destino_Antioquia", "departamento_destino_Narino",
    "departamento_destino_Quindio", "departamento_destino_Tolima",
    "departamento_destino_Valle",
]
# renombra las columnas one-hot para que coincidan con las del modelo
clima_encoded = clima_encoded.rename(columns=lambda c: c.replace("departamento_", "departamento_destino_"))
for col in features_modelo:
    if col not in clima_encoded.columns:
        clima_encoded[col] = 0

X_prod = clima_encoded[features_modelo]

# 4. Predecir
clima_pd["demanda_predicha_tm"] = modelo_prod.predict(X_prod).round(1)

# 5. resultado
resultado = clima_pd[["departamento", "anio", "mes", "precip_pronosticada_mm", "demanda_predicha_tm"]]
print("Predicciones generadas:", len(resultado))
resultado.head(12)

In [0]:
spark.sql("SELECT COUNT(*) FROM gold.prediccion_demanda").show()